# 💓 OPERAÇÃO BATIMENTO
### Simulado Tabular 2 · Trilha de Dados Tabulares · preparação para a OIAA 2026

---

> **"No plantão da noite somos eu e uma enfermeira para doze gestantes no monitor. O
> traçado que eu não olho a tempo é o que me tira o sono."**
>
> — Dra. Helena Duarte, obstetra plantonista

## 📨 Chamado do centro obstétrico

**De:** Dra. Helena Duarte · Centro Obstétrico de uma maternidade pública
**Para:** Equipe de Análise de Dados

> A **cardiotocografia** registra, ao mesmo tempo, o batimento cardíaco do bebê e as
> contrações do útero. O aparelho imprime uma fita; o software extrai dela uma série de
> medidas: a frequência de base, quantas acelerações e desacelerações aconteceram, quão
> variável é o batimento, o formato do histograma dos batimentos.
>
> Temos uma base **real e pública** com 2.126 exames. Cada um foi classificado por **três
> obstetras em consenso** em três estados:
>
> - **Normal** — bebê bem.
> - **Suspeito** — pede atenção e repetição do exame.
> - **Patológico** — pede ação imediata.
>
> Aviso que a base é **muito desequilibrada**, porque a vida é assim: quase oito em cada
> dez exames são normais, e menos de um em dez é patológico. Um sistema que dissesse
> "normal" para tudo acertaria 78% e seria inútil, pior do que inútil: perigoso.
>
> E o erro que eu não posso aceitar tem nome: **patológico classificado como normal**. Esse
> é o traçado que ninguém olha a tempo.
>
> — Dra. Helena Duarte

---

*Cenário simulado; dados reais (Ayres-de-Campos et al., 2000, SisPorto; UCI Machine
Learning Repository). As contagens de eventos foram convertidas para "por 10 minutos". As
colunas do laudo morfológico do especialista foram removidas, porque praticamente
entregariam a resposta.*

## 🎯 A sua missão

Construir um modelo que, a partir das medidas extraídas do exame, aponte o **estado
fetal**: **normal**, **suspeito** ou **patológico**.

Esta operação traz duas novidades em relação à Operação Farol: um desequilíbrio muito
mais forte entre as classes, e variáveis em escalas completamente diferentes (batimentos
por minuto na casa dos 130, variâncias na casa das centenas, eventos por 10 minutos perto
de zero). Isso vai importar na escolha e na preparação do modelo.

### 🗺️ O mapa da operação

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | Descobrir que medidas separam um traçado normal de um patológico | Só clicando |
| **2 · CONFIGURAR** | Escolher modelo, escala e parâmetros | Só arrastando |
| **3 · MODELAR** | Construir, padronizar, treinar e prever | ✏️ Completando **9 lacunas** |
| **4 · AVALIAR** | Contar os patológicos que escaparam | Só clicando |

> ⚠️ **Antes de começar:** vá em `Ambiente de execução` → `Alterar o tipo de ambiente de
> execução` e confirme que está em **CPU**. Tudo aqui roda em segundos, sem placa de vídeo.
>
> Use `Arquivo` → `Salvar uma cópia no Drive` antes de começar, para não perder o trabalho.
>
> 📎 **Simulado de preparação.** Este caderno imita o formato dos desafios oficiais da Fase 1,
> com dados públicos diferentes. O gabarito do teste fica com o corretor
> (`simulados/corrigir.py`), nunca dentro do caderno.

In [ ]:
#@title 🛠️ Preparação: abrindo os exames { display-mode: "form" }
# Os arquivos treino.csv e teste.csv ficam na mesma pasta deste caderno.
URL_TREINO = ""
URL_TESTE = ""

import pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

NOMES = {0: "Normal", 1: "Suspeito", 2: "Patológico"}
CORES = ["#3DCB55", "#FFC107", "#E6177A"]
ALVO = "estado_fetal"
CLASSE_CRITICA = 2

def _carregar(url, nome_local):
    """Usa a cópia ao lado do caderno; senão tenta o endereço; senão pede o envio."""
    if pathlib.Path(nome_local).exists():
        return pd.read_csv(nome_local)
    if url:
        print(f"Baixando {nome_local}...")
        try:
            return pd.read_csv(url)
        except Exception as erro:
            print(f"  não deu certo ({erro.__class__.__name__}).")
    from google.colab import files
    print(f"  envie o arquivo {nome_local}:")
    files.upload()
    return pd.read_csv(nome_local)

dados = _carregar(URL_TREINO, "treino.csv")
_teste = _carregar(URL_TESTE, "teste.csv")
IDS_TESTE = _teste["id_exame"].to_numpy()
dados_teste = _teste.drop(columns=["id_exame"])
N_TESTE = len(dados_teste)

print("Ambiente carregado.\n")
print(f"  Conjunto público (com respostas) .... {len(dados):>5} exames")
print(f"  Conjunto de teste (sem respostas) ... {N_TESTE:>5} exames  [tamanho fixo]")
print(f"  Medidas por exame ................... {dados.shape[1]-1}\n")
for k, n in NOMES.items():
    q = (dados[ALVO] == k).sum()
    print(f"     {n:<11} ({k}) .................. {q:>5}   {q/len(dados)*100:4.1f}%")
print("\n  Guardem essas porcentagens: elas explicam por que a acurácia vai enganar.")
display(dados.head())

---
# 🔍 BLOCO 1 · EXPLORAR
## *Leiam o traçado com os olhos da Dra. Helena*

> **Helena:** *"Um bebê saudável acelera o coração quando se mexe. O batimento de um bebê
> bem é variável, 'vivo'. Traçado liso demais me preocupa mais do que traçado agitado."*

| Grupo de medidas | Variáveis |
|:---|:---|
| Frequência de base | `linha_de_base_bpm` |
| Eventos (por 10 min) | `aceleracoes_10min`, `movimentos_fetais_10min`, `contracoes_10min`, `desaceleracoes_leves_10min`, `desaceleracoes_severas_10min`, `desaceleracoes_prolongadas_10min` |
| Variabilidade | `pct_variab_curta_anormal`, `media_variab_curta`, `pct_variab_longa_anormal`, `media_variab_longa` |
| Histograma dos batimentos | `hist_largura`, `hist_minimo`, `hist_maximo`, `hist_n_picos`, `hist_n_zeros`, `hist_moda`, `hist_media`, `hist_mediana`, `hist_variancia`, `hist_tendencia` |

Transformem a fala da Helena em hipóteses testáveis: *"poucas acelerações indicam
risco"*, *"variabilidade curta anormal por muito tempo indica risco"*. Depois confiram nos
gráficos.

In [ ]:
#@title 📊 Investigue os dados: escolha a variável E o tipo de gráfico { display-mode: "form" }
variavel = "aceleracoes_10min"  #@param ["linha_de_base_bpm", "aceleracoes_10min", "movimentos_fetais_10min", "contracoes_10min", "desaceleracoes_leves_10min", "desaceleracoes_prolongadas_10min", "pct_variab_curta_anormal", "media_variab_curta", "pct_variab_longa_anormal", "media_variab_longa", "hist_largura", "hist_moda", "hist_media", "hist_variancia", "hist_tendencia"]
tipo_de_grafico = "Boxplot"  #@param ["Boxplot", "Histograma sobreposto", "Violino", "Barras de media"]

serie = dados[variavel]
grupos = [serie[dados[ALVO] == k] for k in NOMES]
rot = list(NOMES.values())

fig, ax = plt.subplots(figsize=(8.5, 4.4))
if tipo_de_grafico == "Boxplot":
    bp = ax.boxplot(grupos, patch_artist=True, tick_labels=rot,
                    medianprops=dict(color="black", linewidth=2))
    for p, cor in zip(bp["boxes"], CORES): p.set_facecolor(cor); p.set_alpha(.8)
    ax.set_ylabel(variavel)
elif tipo_de_grafico == "Histograma sobreposto":
    bins = np.histogram_bin_edges(serie, bins=30)
    for g, cor, r in zip(grupos, CORES, rot):
        ax.hist(g, bins=bins, alpha=.55, color=cor, label=r, edgecolor="white", density=True)
    ax.legend(); ax.set_xlabel(variavel); ax.set_ylabel("densidade")
    print("Histograma normalizado (density=True): cada classe soma 1. Sem isso, a")
    print("classe Normal, oito vezes maior, esconderia as outras duas.\n")
elif tipo_de_grafico == "Violino":
    vp = ax.violinplot(grupos, showmedians=True)
    for corpo, cor in zip(vp["bodies"], CORES): corpo.set_facecolor(cor); corpo.set_alpha(.75)
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels(rot); ax.set_ylabel(variavel)
else:
    med = [g.mean() for g in grupos]; err = [g.std()/np.sqrt(len(g)) for g in grupos]
    ax.bar(rot, med, yerr=err, capsize=7, color=CORES, edgecolor="#1F2A44")
    ax.set_ylabel(f"média de {variavel}")
ax.set_title(f"'{variavel}' por estado fetal", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"Escala de '{variavel}': de {serie.min():.2f} a {serie.max():.2f}.")
print("Anotem a escala de algumas variáveis. A diferença entre elas volta no Bloco 2.")

In [ ]:
#@title 🔭 Duas medidas ao mesmo tempo { display-mode: "form" }
eixo_x = "pct_variab_curta_anormal"  #@param ["pct_variab_curta_anormal", "aceleracoes_10min", "pct_variab_longa_anormal", "media_variab_curta", "linha_de_base_bpm", "hist_media"]
eixo_y = "pct_variab_longa_anormal"  #@param ["pct_variab_longa_anormal", "aceleracoes_10min", "pct_variab_curta_anormal", "media_variab_curta", "desaceleracoes_prolongadas_10min", "hist_variancia"]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for k in [0, 1, 2]:
    sub = dados[dados[ALVO] == k]
    ax.scatter(sub[eixo_x], sub[eixo_y], s=14 if k == 0 else 22, alpha=.45 if k == 0 else .8,
               color=CORES[k], label=NOMES[k], edgecolors="none")
ax.set_xlabel(eixo_x); ax.set_ylabel(eixo_y); ax.legend()
ax.set_title("Onde cada estado se concentra", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
print("Existe uma região do gráfico onde só aparecem patológicos? E onde os suspeitos")
print("se misturam com os normais? É ali que o modelo vai errar.")

---
# ⚙️ BLOCO 2 · CONFIGURAR
## *As decisões que definem o desempenho*

| Conjunto | Tamanho | Vocês têm as respostas? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 1.691 exames | Sim | Treinar o modelo e validar as escolhas |
| **Teste** | **423 (fixo)** | Não | Comparação final (corrigida por `simulados/corrigir.py`) |

Os valores iniciais dos controles abaixo são **deliberadamente modestos**.

In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
tamanho_da_validacao = 25  #@param {type:"slider", min:10, max:40, step:5}
modelo_escolhido = "Regressao Logistica"  #@param ["Arvore de Decisao", "Floresta Aleatoria", "Regressao Logistica"]
padronizar = False  #@param {type:"boolean"}
profundidade = 4  #@param {type:"slider", min:2, max:20, step:1}
n_arvores = 100  #@param {type:"slider", min:50, max:600, step:50}
forca_C = 1.0  #@param {type:"slider", min:0.01, max:50, step:0.01}

print("PARÂMETROS DEFINIDOS")
print(f"   Validação .......... {tamanho_da_validacao}%")
print(f"   Modelo ............. {modelo_escolhido}")
print(f"   Padronizar ......... {'sim' if padronizar else 'não'}")
if modelo_escolhido == "Regressao Logistica":
    print(f"   Força C ............ {forca_C}")
else:
    print(f"   Profundidade ....... {profundidade}")
if modelo_escolhido == "Floresta Aleatoria":
    print(f"   Árvores ............ {n_arvores}")
if modelo_escolhido == "Regressao Logistica" and not padronizar:
    print("\n[!] Regressão Logística com variáveis entre 0,01 e 3.000, sem padronização.")
    print("    Observem o resultado e depois liguem 'padronizar'. Só isso, um fator por vez.")

### As decisões e seus efeitos

**Padronização.** O `StandardScaler` subtrai a média e divide pelo desvio-padrão de cada
variável, deixando todas na mesma escala. Modelos lineares e baseados em distância são
sensíveis à escala; árvores não são. E a padronização **estima parâmetros dos dados**
(média e desvio): portanto só pode ser ajustada no treino. Colocá-la dentro de um
`Pipeline` garante isso automaticamente — o `fit` do pipeline ajusta o padronizador só
com os dados que recebe.

**Desequilíbrio forte.** Com 8% de patológicos, o modelo pode "desistir" da classe rara e
ainda parecer ótimo na acurácia. O parâmetro que muda o peso das classes está nas tabelas
abaixo — e aqui ele não é detalhe.

#### 🔧 Padronizador

```python
from sklearn.preprocessing import StandardScaler
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `with_mean` | `True` | Subtrai a média de cada variável |
| `with_std` | `True` | Divide pelo desvio-padrão |

### 📗 Referência técnica: nome exato e parâmetros de cada classe

**Vocês vão precisar desta tabela no Bloco 3.**

> **Por que isso importa.** No Bloco 3, apenas alguns parâmetros aparecem preenchidos.
> Todos os demais assumem silenciosamente o valor padrão. Completar só o nome da classe
> produz um resultado correto e mediano. Estudar esta tabela e **acrescentar parâmetros
> à chamada** produz resultados melhores. A diferença é intencional.

#### 🌳 Árvore de Decisão

```python
from sklearn.tree import DecisionTreeClassifier
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `max_depth` | `None` | Profundidade máxima. `None` cresce até separar tudo — e decora o treino |
| `min_samples_leaf` | `1` | Mínimo de exemplos por folha. Aumentar suaviza as regras |
| `min_samples_split` | `2` | Mínimo de exemplos para tentar um novo corte |
| `criterion` | `'gini'` | Medida de impureza. Alternativa: `'entropy'` |
| `class_weight` | `None` | `'balanced'` dá mais peso às classes raras |
| `random_state` | `None` | Semente, para o resultado ser reproduzível |

#### 🌲 Floresta Aleatória

```python
from sklearn.ensemble import RandomForestClassifier
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `n_estimators` | `100` | Número de árvores |
| `max_depth` | `None` | Profundidade máxima de cada árvore |
| `min_samples_leaf` | `1` | Mínimo de exemplos por folha |
| `max_features` | `'sqrt'` | Variáveis sorteadas em cada corte. Aceita fração |
| `class_weight` | `None` | `'balanced'` ou `'balanced_subsample'` |
| `n_jobs` | `None` | `-1` usa todos os núcleos |
| `random_state` | `None` | Semente |

#### 📈 Regressão Logística

```python
from sklearn.linear_model import LogisticRegression
```

| Parâmetro | Padrão | O que faz |
|:---|:---|:---|
| `C` | `1.0` | Inverso da força da regularização. **Menor regulariza mais** |
| `penalty` | `'l2'` | Tipo de regularização |
| `max_iter` | `100` | Iterações máximas. **Frequentemente insuficiente**: o aviso de não convergência passa despercebido |
| `class_weight` | `None` | `'balanced'` compensa o desequilíbrio entre classes |
| `random_state` | `None` | Semente |

---
# ✏️ BLOCO 3 · MODELAR
## *Padronizar dentro do pipeline*

| Método | Operação | Onde se aplica |
|:---|:---|:---|
| `.fit(X, y)` | **Ajusta** o pipeline inteiro: primeiro o padronizador, depois o modelo | Só no treino |
| `.predict(X)` | Padroniza com a média/desvio **do treino** e **infere** a classe | Validação e teste |

`make_pipeline(A, B)` encadeia um transformador `A` e um modelo `B` num único objeto,
que se comporta como um modelo comum.

> ⚠️ **A célula não executa enquanto houver lacunas.**

In [ ]:
# ✏️ SUA VEZ · Parte 1: separar os dados e construir o modelo
X = dados.drop(columns=[ALVO])
y = dados[ALVO]

# LACUNA 1 ▸ que vetor precisa ter a proporção preservada nas duas partes?
#            Com 8% de patológicos, errar aqui pode deixar a validação quase sem eles.
X_treino, X_valida, y_treino, y_valida = train_test_split(
    X, y, test_size=tamanho_da_validacao/100, random_state=42, stratify=____)

# LACUNAS 2 a 6 ▸ nomes exatos das classes (referência do Bloco 2) e a cadeia condicional.
if modelo_escolhido == "Arvore de Decisao":
    nucleo = ____(max_depth=profundidade, random_state=42)

____ modelo_escolhido == "Floresta Aleatoria":
    nucleo = ____(n_estimators=n_arvores, max_depth=profundidade,
                  random_state=42, n_jobs=-1)

____:
    nucleo = ____(C=forca_C, random_state=42)

# LACUNA 7 ▸ qual transformador coloca todas as medidas na mesma escala?
#            O nome está na referência do Bloco 2.
modelo = make_pipeline(____(), nucleo) if padronizar else nucleo

# ZONA LIVRE ▸ acrescentem parâmetros às chamadas acima. Nesta operação, um deles
#   muda o jogo para a classe rara, e outro (só na Regressão Logística) evita que o
#   treino pare antes de convergir. Um de cada vez, registrando o efeito no Bloco 4.

print(f"Treino: {len(X_treino)} exames  |  Validação: {len(X_valida)} exames")
print(f"Patológicos na validação: {(y_valida == 2).sum()}")
print(f"Modelo: {type(nucleo).__name__}{' dentro de um Pipeline com StandardScaler' if padronizar else ''}")

In [ ]:
# ✏️ SUA VEZ · Parte 2: treinar e prever
# LACUNA 8 ▸ qual método ajusta o modelo (e o padronizador, se houver) ao treino?
modelo.____(X_treino, y_treino)

# LACUNA 9 ▸ qual método infere o estado de exames que o modelo nunca viu?
previsoes = modelo.____(X_valida)

print(f"Modelo ajustado. {len(previsoes)} previsões geradas para a validação.")

In [ ]:
#@title 🔎 Verificação das lacunas (inclusive vazamento na padronização) { display-mode: "form" }
erros = []
try:
    esperado = np.bincount(y, minlength=3) * len(y_treino) / len(y)
    if np.abs(np.bincount(y_treino, minlength=3) - esperado).max() > 1.5:
        erros.append("LACUNA 1: a proporção de patológicos no treino difere do conjunto "
                     "completo. O 'stratify' não recebeu o vetor correto.")
except NameError:
    erros.append("LACUNA 1: X_treino/y_treino não existem.")
try:
    esperado = {"Arvore de Decisao": "DecisionTreeClassifier",
                "Floresta Aleatoria": "RandomForestClassifier",
                "Regressao Logistica": "LogisticRegression"}[modelo_escolhido]
    if type(nucleo).__name__ != esperado:
        erros.append(f"LACUNAS 2 a 6: para '{modelo_escolhido}' era esperada a classe "
                     f"{esperado}, mas foi construída {type(nucleo).__name__}.")
except NameError:
    erros.append("LACUNAS 2 a 6: a variável 'nucleo' não existe.")
if padronizar:
    try:
        passos = dict(modelo.steps)
        if "standardscaler" not in passos:
            erros.append("LACUNA 7: 'padronizar' está ligado, mas o pipeline não contém "
                         "um StandardScaler.")
        elif hasattr(passos["standardscaler"], "mean_"):
            # O padronizador precisa ter visto SÓ o treino.
            if not np.allclose(passos["standardscaler"].mean_, X_treino.mean().values):
                erros.append("LACUNA 7: VAZAMENTO. A média do padronizador não é a do "
                             "treino: ele foi ajustado com dados que incluem a validação.")
    except AttributeError:
        erros.append("LACUNA 7: o pipeline não foi construído corretamente.")
try:
    final = modelo.steps[-1][1] if hasattr(modelo, "steps") else modelo
    if not hasattr(final, "classes_"):
        erros.append("LACUNA 8: o modelo não possui parâmetros ajustados.")
except NameError: pass
try:
    if len(previsoes) != len(X_valida):
        erros.append(f"LACUNA 9: {len(previsoes)} previsões para {len(X_valida)} casos.")
except NameError:
    erros.append("LACUNA 9: a variável 'previsoes' não existe.")

if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
else:
    print("VERIFICAÇÃO APROVADA\n")
    extras = {k: v for k, v in nucleo.get_params().items()
              if k in ("class_weight", "min_samples_leaf", "max_features", "max_iter",
                       "criterion", "min_samples_split")
              and v not in (None, 1, 2, "gini", "sqrt", 100)}
    if extras:
        print(f"   Parâmetros adicionais detectados na ZONA LIVRE: {extras}")
    else:
        print("   Nenhum parâmetro adicional declarado. Com 8% de patológicos, as tabelas")
        print("   do Bloco 2 têm pelo menos um parâmetro que vocês deveriam testar.")

---
# 📈 BLOCO 4 · AVALIAR
## *Quantos patológicos escaparam?*

> **Helena:** *"Me digam primeiro quantos patológicos o sistema deixou passar como normais.
> Depois conversamos sobre o resto."*

A métrica oficial é o **F1-macro**. A de apoio é a **revocação da classe patológica**: dos
exames que eram patológicos, quantos o modelo sinalizou como patológicos.

E a leitura da matriz aqui é direcional: um *suspeito* chamado de *patológico* gera uma
reavaliação desnecessária; um *patológico* chamado de *normal* pode custar uma vida. As
células fora da diagonal **não valem o mesmo**.

In [ ]:
#@title 🏆 Desempenho na validação { display-mode: "form" }
acuracia = accuracy_score(y_valida, previsoes)
f1 = f1_score(y_valida, previsoes, average="macro")
recall_patologico = recall_score(y_valida, previsoes, labels=[2], average="macro")
precisao_patologico = precision_score(y_valida, previsoes, labels=[2], average="macro",
                                      zero_division=0)

print("="*62)
print("             DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*62)
print(f"  Acurácia ............................... {acuracia*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) ........... {f1*100:5.1f}%")
print(f"  🔴 Revocação do PATOLÓGICO .............. {recall_patologico*100:5.1f}%")
print(f"     Precisão do patológico ............... {precisao_patologico*100:5.1f}%")
print("="*62)
print(f"  Referência: 'normal' para todos daria {(y_valida==0).mean()*100:.1f}% de acurácia")
print("  e F1-macro de 29%.")
if   f1 >= .85: print("\n  Desempenho alto. Olhem agora a linha dos patológicos na matriz.")
elif f1 >= .80: print("\n  Desempenho sólido. Ainda há espaço no peso das classes e no modelo.")
elif f1 >= .74: print("\n  Desempenho intermediário. Padronização e profundidade merecem teste.")
else:           print("\n  Desempenho baixo. O modelo mal enxerga as classes raras.")

cm = confusion_matrix(y_valida, previsoes, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=list(NOMES.values())).plot(
    ax=ax, cmap="viridis", colorbar=False)
ax.set_title("Matriz de confusão (validação)", fontweight="bold")
ax.set_xlabel("O que o modelo DISSE"); ax.set_ylabel("O que REALMENTE era")
plt.tight_layout(); plt.show()

print("DECOMPOSIÇÃO DOS ERROS QUE IMPORTAM:\n")
print(f"   🔴 PATOLÓGICOS chamados de NORMAL ......... {cm[2,0]}   ← o erro inaceitável")
print(f"   🟠 PATOLÓGICOS chamados de SUSPEITO ....... {cm[2,1]}")
print(f"   🟡 SUSPEITOS chamados de NORMAL ........... {cm[1,0]}")
print(f"   ⚪ NORMAIS sinalizados como patológicos ... {cm[0,2]}   ← alarme falso\n")
print(classification_report(y_valida, previsoes, labels=[0, 1, 2],
      target_names=list(NOMES.values()), digits=3, zero_division=0))

In [ ]:
#@title 🔦 O que o modelo está enxergando? { display-mode: "form" }
final = modelo.steps[-1][1] if hasattr(modelo, "steps") else modelo
if hasattr(final, "feature_importances_"):
    imp = pd.Series(final.feature_importances_, index=X.columns).sort_values()
    titulo = "Importância de cada medida"
else:
    imp = pd.Series(np.abs(final.coef_).mean(axis=0), index=X.columns).sort_values()
    titulo = "Peso de cada medida (média |coeficiente|)"
    if not padronizar:
        titulo += " — SEM padronização: pesos incomparáveis"
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp.index, imp.values, color="#E6177A", edgecolor="#0A0F2C")
ax.set_title(titulo, fontsize=12, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
print("AS CINCO MEDIDAS DE MAIOR PESO:\n")
for i, (nome, val) in enumerate(imp.sort_values(ascending=False).head(5).items(), 1):
    print(f"   {i}. {nome}  ({val:.3f})")
print("\nComparem com as hipóteses do Bloco 1 e com a fala da Dra. Helena sobre")
print("acelerações e variabilidade. O modelo chegou ao mesmo lugar que a clínica?")

---
# 🤔 PARADA OBRIGATÓRIA: quem decide no plantão

> **Helena:** *"Se o sistema apitar para todo traçado duvidoso, em uma semana a equipe
> para de ouvir o apito. Se ele apitar pouco, a gente confia e deixa de olhar. Os dois
> caminhos terminam no mesmo lugar."*

### Questões para a equipe deliberar

1. **Fadiga de alarme.** Aumentar o peso da classe patológica sobe a revocação e também o
   número de alarmes falsos. Quantos alarmes falsos por patológico detectado a equipe do
   plantão tolera antes de parar de confiar no sistema? Usem os números da matriz de vocês.

2. **O que o rótulo mede.** O rótulo é a opinião de três obstetras olhando **o mesmo
   traçado** — não o que aconteceu com o bebê depois. O modelo, então, aprende a imitar
   especialistas. Que limite isso impõe? Em que situação imitar bem o especialista não é
   suficiente?

3. **Outro aparelho, outra maternidade.** Os exames vêm de um hospital português dos anos
   1990. Um aparelho diferente pode calcular a variabilidade de outro jeito. Como vocês
   verificariam se o modelo ainda vale antes de usá-lo aqui?

4. **Regime de uso.** Autônomo (decide sozinho) ou assistivo (prioriza a fila para a
   médica)? Quem responde se um patológico escapar?

In [ ]:
#@title 📝 O relatório da sua equipe para o centro obstétrico { display-mode: "form" }
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. Qual modelo e configuração vocês escolheram? O que a padronização mudou?**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. Quantos patológicos escaparam como normais, e o que vocês fizeram para reduzir isso?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. Quantos alarmes falsos isso custou? Qual regime de uso vocês recomendam?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. Como verificar se o modelo vale num aparelho ou hospital diferente?**
resposta_4 = ""  #@param {type:"string"}

relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido,
             "padronizar": padronizar, "validacao_pct": tamanho_da_validacao,
             "profundidade": profundidade, "n_arvores": n_arvores, "C": forca_C,
             "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "recall_patologico": round(recall_patologico, 4),
             "precisao_patologico": round(precisao_patologico, 4),
             "just_modelo": resposta_1, "just_erros_criticos": resposta_2,
             "just_regime": resposta_3, "just_outro_hospital": resposta_4}
faltando = [k for k in ["equipe", "just_modelo", "just_erros_criticos", "just_regime",
                        "just_outro_hospital"] if not str(relatorio[k]).strip()]
if faltando:
    print("⚠️ Ainda faltam campos:", ", ".join(faltando))
else:
    print("Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
import json, pathlib

PASTA_SUBMISSAO = pathlib.Path("submissoes")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido, "f1_macro": round(f1, 4)}

# O pipeline padroniza o teste com a média e o desvio do TREINO: só predict, nada de fit.
palpites = modelo.predict(dados_teste[X.columns])
submissao = pd.DataFrame({"id_exame": IDS_TESTE, "estado_previsto": palpites})

assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_batimento.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_batimento.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_batimento.csv   ({len(submissao)} linhas)")
print("  relatorio_batimento.json")
print("="*60)
print("\n  Distribuição prevista no teste:")
for k, v in pd.Series(palpites).value_counts().sort_index().items():
    print(f"     {NOMES[k]:<11} {v:>4}  ({v/len(palpites)*100:.1f}%)")
print(f"  No conjunto público, os patológicos são {(dados[ALVO]==2).mean()*100:.1f}%.")
display(submissao.head())
print("\n  Para corrigir (da raiz do repositório):")
print(f"     python simulados/corrigir.py batimento {csv_saida}")

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA OPERAÇÃO BATIMENTO

> **Helena:** *"Colocamos o sistema no plantão como fila de prioridade: ele reordena os
> traçados e eu olho de cima para baixo. Nenhum traçado deixa de ser olhado; os graves são
> olhados primeiro. Vocês me entregaram, junto com o modelo, o número de alarmes falsos
> que ele custa. Foi isso que convenceu a equipe."*

---

### Competências praticadas neste simulado

- Desequilíbrio forte de classes e por que a acurácia engana
- `class_weight` e o compromisso entre revocação da classe rara e alarmes falsos
- Sensibilidade à escala e `StandardScaler`
- `Pipeline` como proteção contra vazamento na padronização
- Leitura direcional da matriz de confusão com custos diferentes por célula
- O rótulo como opinião de especialista, não como desfecho
- Fadiga de alarme e regimes autônomo × assistivo